In [1]:
import pandas as pd

In [2]:
price = pd.read_csv("./data/raw/price.csv")
sentiment = pd.read_csv("./data/raw/sentiment.csv").drop(columns='Accurate Sentiments', axis=1)

In [3]:
price.head()

,Date,Open,High,Low,Close,Volume
0,2021-11-05,61460.078125,62541.468750,60844.609375,61125.675781,30605102446
1,2021-11-06,61068.875000,61590.683594,60163.781250,61527.480469,29094934221
2,2021-11-07,61554.921875,63326.988281,61432.488281,63326.988281,24726754302
3,2021-11-08,63344.066406,67673.742188,63344.066406,67566.828125,41125608330
4,2021-11-09,67549.734375,68530.335938,66382.062500,66971.828125,42357991721


In [4]:
sentiment.head()

,Date,Short Description
0,2021-11-05 04:42:00,Bitcoin price is consolidating near the USD 62...
1,2021-11-05 08:15:00,Congress could finally approve or reject the m...
2,2021-11-05 10:24:00,Bitcoin increasingly becoming a political inst...
3,2021-11-05 16:58:00,There is still potential for the price of bitc...
4,2021-11-05 21:00:00,'Several companies' are looking to Latin Ameri...


In [5]:
price["Date"] = pd.to_datetime(price["Date"])

In [6]:
sentiment["Date"] = pd.to_datetime(sentiment["Date"]).dt.normalize()

In [7]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

sentiment["sentiment_score"] = sentiment["Short Description"].apply(
    lambda x: analyzer.polarity_scores(str(x))["compound"]
)

df_sentiment = (
    sentiment.groupby("Date")["sentiment_score"]
    .mean()
    .reset_index()
)

print(df_sentiment.head())

        Date  sentiment_score
0 2021-11-05         0.214960
1 2021-11-06         0.000000
2 2021-11-08        -0.155775
3 2021-11-09         0.186925
4 2021-11-10         0.155275


In [8]:
df = price.merge(df_sentiment, on="Date", how="left")

df["sentiment_score"] = df["sentiment_score"].fillna(0)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1043 entries, 0 to 1042
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             1043 non-null   datetime64[ns]
 1   Open             1043 non-null   float64       
 2   High             1043 non-null   float64       
 3   Low              1043 non-null   float64       
 4   Close            1043 non-null   float64       
 5   Volume           1043 non-null   int64         
 6   sentiment_score  1043 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(1)
memory usage: 57.2 KB


In [9]:
df.head(10)

,Date,Open,High,Low,Close,Volume,sentiment_score
0,2021-11-05,61460.078125,62541.468750,60844.609375,61125.675781,30605102446,0.214960
1,2021-11-06,61068.875000,61590.683594,60163.781250,61527.480469,29094934221,0.000000
2,2021-11-07,61554.921875,63326.988281,61432.488281,63326.988281,24726754302,0.000000
3,2021-11-08,63344.066406,67673.742188,63344.066406,67566.828125,41125608330,-0.155775
4,2021-11-09,67549.734375,68530.335938,66382.062500,66971.828125,42357991721,0.186925
5,2021-11-10,66953.335938,68789.625000,63208.113281,64995.230469,48730828378,0.155275
6,2021-11-11,64978.890625,65579.015625,64180.488281,64949.960938,35880633236,0.043900
7,2021-11-12,64863.980469,65460.816406,62333.914062,64155.941406,36084893887,0.047375
8,2021-11-13,64158.121094,64915.675781,63303.734375,64469.527344,30474228777,0.000000
9,2021-11-14,64455.371094,65495.179688,63647.808594,65466.839844,25122092191,0.400500


In [10]:
df.to_csv("./data/processed/processed_data.csv", index=False)

In [12]:
price["Close"].shift(-1).tail()

1038    57019.535156
1039    57648.710938
1040    57343.171875
1041    58127.011719
1042             NaN
Name: Close, dtype: float64